# 🧪 Exercise 2 – News API + Word Frequency + Word Similarity (gensim)
Steps:
1. Fetch news articles from NewsAPI (technology topic)
2. Extract text (title + description)
3. Clean the text (lowercase, remove digits/punctuation, etc.)
4. Tokenize and remove stopwords
5. Find most frequent words
6. Train a Word2Vec model using gensim
7. Compute:
   - Similar words to a given word
   - Similarity between pairs of words


In [1]:
!pip install requests gensim spacy nltk
!python -m spacy download en_core_web_sm
import nltk
nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 110.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [27]:
API_KEY = "YOUR_API_KEY_HERE"
base_url = "https://newsapi.org/v2/everything"

In [20]:
#1
import requests
#https://newsapi.org/register -> for api key
#👉 https://newsapi.org/docs/endpoints/everything
params = {
    "q": "technology",
    "language": "en",
    "pageSize": 50,
    "sortBy": "publishedAt",
    "apiKey": API_KEY,
}
try:
    res = requests.get(base_url, params=params, timeout=10)
    res.raise_for_status()
    data = res.json()
except requests.exceptions.HTTPError as e:
    print("❌ HTTP Error:", e)
except requests.exceptions.ConnectionError:
    print("❌ Connection Error – check your internet!")
except requests.exceptions.Timeout:
    print("❌ Timeout Error!")
except Exception as e:
    print("❌ Unexpected Error:", e)
else:
    print("✅ Request successful!")

✅ Request successful!


In [21]:
data

{'status': 'ok',
 'totalResults': 47272,
 'articles': [{'source': {'id': None, 'name': 'pymnts.com'},
   'author': 'PYMNTS',
   'title': 'Verra Mobility Debuts In-Vehicle Commerce Platform AutoKinex',
   'description': 'Verra Mobility has introduced AutoKinex, calling its “OEM-ready” in-vehicle commerce platform. The platform integrates payment technology for mobility services such as tolls, parking, fueling and electric vehicle charging, the company announced Wednesday (Nov…',
   'url': 'https://www.pymnts.com/connectedeconomy/2025/verra-mobility-debuts-in-vehicle-commerce-platform-autokinex/',
   'urlToImage': 'https://www.pymnts.com/wp-content/uploads/2025/11/Ram-truck-crop.jpg',
   'publishedAt': '2025-11-20T14:20:30Z',
   'content': 'Verra Mobility has introduced AutoKinex, calling its OEM-ready in-vehicle commerce platform.The platform integrates payment technology for mobility services such as tolls, parking, fueling and electr… [+2796 chars]'},
  {'source': {'id': None, 'name':

In [22]:
# 2
articles = data.get("articles", []) # data['article]  لیست خبرهاست -> اگر نبود لیست خالی برگردونه
texts = []
for art in articles:
    title = art.get("title") or ""
    desc = art.get("description") or ""
    text = f"{title}. {desc}".strip()
    if text:
        texts.append(text)
print("📰 number of articles:", len(texts))
print("\nSample article text:\n", texts[0] if texts else "No articles found")

📰 number of articles: 47

Sample article text:
 Verra Mobility Debuts In-Vehicle Commerce Platform AutoKinex. Verra Mobility has introduced AutoKinex, calling its “OEM-ready” in-vehicle commerce platform. The platform integrates payment technology for mobility services such as tolls, parking, fueling and electric vehicle charging, the company announced Wednesday (Nov…


In [23]:
# 3
import re
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
cleaned_texts = [clean_text(t) for t in texts]

print("Cleaned sample:\n", cleaned_texts[0][:300] if cleaned_texts else "No data")


Cleaned sample:
 verra mobility debuts in vehicle commerce platform autokinex verra mobility has introduced autokinex calling its oem ready in vehicle commerce platform the platform integrates payment technology for mobility services such as tolls parking fueling and electric vehicle charging the company announced w


In [24]:
# 4
import spacy
from nltk.corpus import stopwords
from gensim.models import Word2Vec
nlp = spacy.load("en_core_web_sm")
eng_stopwords = set(stopwords.words('english'))

sentences = []
for txt in cleaned_texts:
    doc = nlp(txt)
    tokens = [
        token.text
        for token in doc
        if not token.is_space
        and not token.is_punct
        and token.text not in eng_stopwords
        and len(token.text) > 2
    ]
    if len(tokens) > 3:
        sentences.append(tokens)

print("number of sentences (articles used):", len(sentences))
print("Sample tokenized sentence:\n", sentences[0] if sentences else "No sentences")


number of sentences (articles used): 47
Sample tokenized sentence:
 ['verra', 'mobility', 'debuts', 'vehicle', 'commerce', 'platform', 'autokinex', 'verra', 'mobility', 'introduced', 'autokinex', 'calling', 'oem', 'ready', 'vehicle', 'commerce', 'platform', 'platform', 'integrates', 'payment', 'technology', 'mobility', 'services', 'tolls', 'parking', 'fueling', 'electric', 'vehicle', 'charging', 'company', 'announced', 'wednesday', 'nov']


In [11]:
# 5
from collections import Counter
all_tokens = [token for sent in sentences for token in sent]

word_freq = Counter(all_tokens)
top_words = word_freq.most_common(30)

print("🔥 Top 30 frequent words:\n")
for w, c in top_words:
    print(f"{w:15} {c}")


🔥 Top 30 frequent words:

company         12
leica           9
new             8
apple           6
next            6
security        6
solutions       6
health          6
monochrom       6
global          6
study           6
report          5
world           5
picus           5
recognized      5
growth          5
cancer          5
market          5
embedded        5
human           5
air             4
canada          4
exposure        4
validation      4
nov             4
technology      4
three           4
better          4
intelligence    4
decision        4


In [13]:
# 6
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,   # کلمه‌هایی که حداقل 2 بار تکرار شدند
    workers=4,
    sg=0           # CBOW (0) یا Skip-gram (1)
)

print("✅ Word2Vec model trained!")
print("Vocabulary size:", len(model.wv.index_to_key))


✅ Word2Vec model trained!
Vocabulary size: 301


In [25]:
# 7
def show_similar(word, topn=10):
    if word not in model.wv.key_to_index:
        print(f"⚠️ '{word}' not in vocabulary")
        return
    print(f"🔍 Most similar to '{word}':\n")
    for w, sim in model.wv.most_similar(word, topn=topn):
        print(f"{w:15} {sim:.3f}")

show_similar("next")
print()
show_similar("better")
show_similar("google")

🔍 Most similar to 'next':

visualization   0.312
customers       0.257
firm            0.251
photo           0.240
music           0.238
company         0.221
dynamic         0.218
documentary     0.208
research        0.190
spread          0.184

🔍 Most similar to 'better':

stoves          0.239
silicon         0.228
hub             0.216
arc             0.200
platforms       0.200
human           0.192
skye            0.191
would           0.189
fifa            0.188
research        0.187
⚠️ 'google' not in vocabulary


In [26]:
def word_similarity(w1, w2):
    if w1 not in model.wv.key_to_index:
        print(f"⚠️ '{w1}' not in vocabulary")
        return
    if w2 not in model.wv.key_to_index:
        print(f"⚠️ '{w2}' not in vocabulary")
        return
    sim = model.wv.similarity(w1, w2)
    print(f"📏 similarity('{w1}', '{w2}') = {sim:.3f}")

word_similarity("next", "visualization")
word_similarity("hub", "arc")

📏 similarity('next', 'visualization') = 0.312
📏 similarity('hub', 'arc') = -0.123
⚠️ 'robot' not in vocabulary


In [16]:
model.wv.key_to_index

{'company': 0,
 'leica': 1,
 'new': 2,
 'study': 3,
 'global': 4,
 'monochrom': 5,
 'health': 6,
 'solutions': 7,
 'security': 8,
 'next': 9,
 'apple': 10,
 'human': 11,
 'embedded': 12,
 'market': 13,
 'cancer': 14,
 'growth': 15,
 'recognized': 16,
 'picus': 17,
 'world': 18,
 'report': 19,
 'skye': 20,
 'summary': 21,
 'author': 22,
 'climate': 23,
 'model': 24,
 'first': 25,
 'startup': 26,
 'think': 27,
 'decision': 28,
 'intelligence': 29,
 'better': 30,
 'three': 31,
 'technology': 32,
 'nov': 33,
 'validation': 34,
 'exposure': 35,
 'canada': 36,
 'air': 37,
 'stage': 38,
 'investors': 39,
 'braun': 40,
 'saas': 41,
 'infragistics': 42,
 'law': 43,
 'dynamic': 44,
 'bat': 45,
 'action': 46,
 'threat': 47,
 'measles': 48,
 'activity': 49,
 'structures': 50,
 'rna': 51,
 'pseudoknots': 52,
 'firm': 53,
 'system': 54,
 'gps': 55,
 'year': 56,
 'change': 57,
 'cup': 58,
 'fifa': 59,
 'supply': 60,
 'resellers': 61,
 'vibe': 62,
 'ctv': 63,
 'appeared': 64,
 'post': 65,
 'documentar